# カテゴリ変数の数値化
性別や乗船した港のような文字列の列を、モデルに入れられる数値の列に変えます。2値の変数は 0 / 1 に、3つ以上の値をとる変数は「ワンホットエンコーディング」で複数の 0 / 1 の列に変えます。学習用と予測用で列がそろっているかの確認方法も学びます。

## 学習目標
- 2値のカテゴリ変数を 0 / 1 の数値に変えられる
- `pd.get_dummies` でワンホットエンコーディングができる
- カテゴリに番号を振るだけ(1, 2, 3)では問題がある場合を説明できる
- 学習用と予測用で、作られる列をそろえられる(`reindex`)

## 2値の変数:0 / 1 にする

性別は female と male の2値なので、「女性なら 1、男性なら 0」の列にします。比較 `==` で True / False を作り、`astype(int)` で 1 / 0 に変えます。

In [1]:
import pandas as pd

# 学習用のデータを読み込む
train = pd.read_csv("train.csv")

# 女性なら True、男性なら False → astype(int) で 1 / 0 に変える
train["Sex"] = (train["Sex"] == "female").astype(int)

# 値ごとの人数を確認する(1 = 女性、0 = 男性)
print(train["Sex"].value_counts())

Sex
0    586
1    305
Name: count, dtype: int64


## 3つ以上の値:ワンホットエンコーディング

乗船した港(C / Q / S)に「C = 1、Q = 2、S = 3」と番号を振ると、モデルは「S は C の 3 倍」「Q は C と S の中間」のように、本来ない大小関係があるものとして扱ってしまうことがあります。

そこで、値ごとに「その値なら 1、そうでなければ 0」の列を作ります。これを **ワンホットエンコーディング** といいます。pandas では `pd.get_dummies` で作れます。

In [2]:
# 欠損値は前の単元と同じく最頻値 S で埋めておく
embarked = train["Embarked"].fillna("S")

# 港ごとに 0 / 1 の列を作る(prefix は列名の先頭、dtype=int で 0 / 1 の整数にする)
embarked_dummies = pd.get_dummies(embarked, prefix="Embarked", dtype=int)

# 元の値と、作られた列を並べて先頭 5 行を確認する
pd.concat([embarked, embarked_dummies], axis=1).head(5)

  Embarked  Embarked_C  Embarked_Q  Embarked_S
0        C           1           0           0
1        S           0           0           1
2        S           0           0           1
3        S           0           0           1
4        S           0           0           1


各行で、乗船した港の列だけが 1 になっています。作った列は `pd.concat(..., axis=1)` で元の表に横につなげて使います。

**客室の等級 `Pclass` はどうするか**:等級(1 等・2 等・3 等)には「1 等が最も上」という順序があり、数値の大小が意味を持つので、この講座では数値のまま使います。順序のないカテゴリ(港・敬称など)はワンホットエンコーディングにします。

## 学習用と予測用で列をそろえる

`get_dummies` は、**データに含まれている値** の列しか作りません。予測用のデータにない値があると、列の数が学習用と変わってしまいます。例として、予測用のデータの一部(先頭 2 人)だけで作ってみます。

In [3]:
# 予測用のデータを読み込む
test = pd.read_csv("test.csv")

# 先頭 2 人の港を確認する(Q と S だけで、C の人はいない)
print(list(test["Embarked"].head(2)))

# 先頭 2 人だけでワンホットエンコーディングすると、C の列が作られない
part = pd.get_dummies(test["Embarked"].head(2), prefix="Embarked", dtype=int)
print(list(part.columns))

['Q', 'S']
['Embarked_Q', 'Embarked_S']


学習用の列(`Embarked_C`・`Embarked_Q`・`Embarked_S`)とそろえるには、`reindex(columns=学習用の列名, fill_value=0)` を使います。足りない列は 0 で追加され、列の順番も学習用と同じになります。

In [4]:
# 学習用の列名に合わせて、足りない列を 0 で追加し、順番もそろえる
part_aligned = part.reindex(columns=embarked_dummies.columns, fill_value=0)
print(list(part_aligned.columns))

['Embarked_C', 'Embarked_Q', 'Embarked_S']


## 演習

予測用のデータ `test.csv` の性別 `Sex` を、学習用と同じ方法(女性 = 1、男性 = 0)で数値に変え、値ごとの人数を表示してください。

In [ ]:
# @starter
import pandas as pd

# 予測用のデータを読み込む
test = pd.read_csv("test.csv")

# 女性なら 1、男性なら 0 の数値に変える(____ に、女性かどうかを判定する比較の式を書く)
test["Sex"] = (____).astype(int)

# 値ごとの人数を表示する
print(test["Sex"].value_counts())

In [5]:
# @solution
# @title: 予測用データの性別を数値にする
# @hint: test["Sex"] == "female" で、女性なら True、男性なら False になります。
# @explanation: 予測用のデータでは、男性が 257 人(0)、女性が 161 人(1)です。学習用と同じく「女性 = 1」にそろえることが大切です。
import pandas as pd

# 予測用のデータを読み込む
test = pd.read_csv("test.csv")

# 女性なら True、男性なら False → astype(int) で 1 / 0 に変える
test["Sex"] = (test["Sex"] == "female").astype(int)

# 値ごとの人数を表示する
print(test["Sex"].value_counts())

Sex
0    257
1    161
Name: count, dtype: int64


学習用のデータで、名前から取り出した敬称 `Title` をワンホットエンコーディングし(列名の先頭は `Title`)、**作られた列の数** と、**人数の多い敬称の列 4 つの合計**(各列の 1 の数)を表示してください。

In [ ]:
# @starter
import pandas as pd

# 学習用のデータを読み込み、名前から敬称を取り出す
train = pd.read_csv("train.csv")
train["Title"] = train["Name"].str.extract(r",\s*([^.]+)\.")[0]

# 敬称ごとに 0 / 1 の列を作る(____ に、ワンホットエンコーディングをする pandas の関数の名前を入れる)
title_dummies = pd.____(train["Title"], prefix="Title", dtype=int)

# 作られた列の数を表示する
print(len(title_dummies.columns))

# 各列の 1 の数(= その敬称の人数)を求め、多い順に 4 つ表示する
print(title_dummies.sum().sort_values(ascending=False).head(4))

In [6]:
# @solution
# @title: 敬称のワンホットエンコーディング
# @hint: pd.get_dummies(列, prefix="Title", dtype=int) で、敬称ごとの 0 / 1 の列が作れます。
# @explanation: 敬称は 16 種類あるので、16 列が作られます。人数の少ない敬称の列はほとんどが 0 で、モデルが学べることもわずかです。次の単元で、少ない敬称を「Rare」にまとめてから列を作ります。
import pandas as pd

# 学習用のデータを読み込み、名前から敬称を取り出す
train = pd.read_csv("train.csv")
train["Title"] = train["Name"].str.extract(r",\s*([^.]+)\.")[0]

# 敬称ごとに 0 / 1 の列を作る
title_dummies = pd.get_dummies(train["Title"], prefix="Title", dtype=int)

# 作られた列の数を表示する
print(len(title_dummies.columns))

# 各列の 1 の数(= その敬称の人数)を求め、多い順に 4 つ表示する
print(title_dummies.sum().sort_values(ascending=False).head(4))

16
Title_Mr        525
Title_Miss      164
Title_Mrs       134
Title_Master     41
dtype: int64


<!-- @question
type: choice
answer: 港の間に本来ない大小関係(S は C の 3 倍など)があるものとしてモデルが扱うおそれがある。
title: 番号を振るだけの問題点
hint: 港の C・Q・S には、大きい・小さいという順序がありません。
explanation: 番号を振るだけだと、モデルはその数値の大小や間隔に意味があるものとして扱うおそれがあります。港のように順序のないカテゴリは、ワンホットエンコーディングで値ごとの 0 / 1 の列にします。等級のように順序に意味がある変数は、数値のまま使うこともできます。
-->
乗船した港(C / Q / S)を、「C = 1、Q = 2、S = 3」と番号を振って数値にした場合の問題点として最も適切なものを選べ。

- 欠損値が増えてしまう。
- 港の間に本来ない大小関係(S は C の 3 倍など)があるものとしてモデルが扱うおそれがある。
- 列の数が増えすぎて、モデルの学習ができなくなる。
- 文字列のままなので、モデルに入れられない。

## まとめ

- 2値のカテゴリ変数は、比較(`==`)と `astype(int)` で 0 / 1 にする。
- 順序のないカテゴリ変数は、`pd.get_dummies` でワンホットエンコーディングする。順序に意味のある等級は数値のまま使う。
- `get_dummies` はデータにある値の列しか作らないので、`reindex(columns=学習用の列名, fill_value=0)` で学習用と予測用の列をそろえる。